# MINspire 2026 Capstone — Predicting House Prices in the Kathmandu Valley
**An Exploratory Data Analysis & Regression Study of the Nepali Housing Market**

Team Name: Team Sunway 

This notebook cleans the raw Nepali Housing Price dataset, explores it, and trains regression
models to predict house price. Place `2020-4-27.csv` in the same folder and run top to bottom.

In [ ]:
import pandas as pd, numpy as np, re
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

%matplotlib inline
plt.rcParams.update({'figure.dpi':110,'axes.grid':True,'grid.alpha':0.25,
                     'axes.spines.top':False,'axes.spines.right':False})
TEAL,CORAL,NAVY,GOLD='#0d7a6f','#e8654f','#22304a','#d9a441'

In [ ]:
raw = pd.read_csv("2020-4-27.csv")
print("Shape:", raw.shape)
raw.head(3)

In [ ]:
raw.isna().sum() 

In [ ]:
SQFT={'ropani':5476.0,'aana':342.25,'paisa':85.5625,'daam':21.390625,
      'sqft':1.0,'sqm':10.7639,'bigha':72900.0,'kattha':3645.0,'dhur':182.25}

def parse_area(s):
    s=str(s).strip()
    if s.lower() in ('nan','') or 'dont know' in s.lower(): return np.nan
    low=s.lower()
    if   'meter'  in low: base='sqm'
    elif 'feet'   in low: base='sqft'
    elif 'ropani' in low: base='ropani'
    elif 'aana'   in low: base='aana'
    elif 'kattha' in low: base='kattha'
    elif 'dhur'   in low: base='dhur'
    elif 'bigha'  in low: base='bigha'
    elif 'paisa'  in low: base='paisa'
    elif 'haat'   in low:
        nums=re.findall(r'[\d.]+',s)
        return float(nums[0])*float(nums[1])*2.25 if len(nums)>=2 else np.nan
    else: return np.nan
    numpart=re.sub(r'[A-Za-z.]','',s).replace('Sq','')
    parts=[p for p in re.split(r'[-/]', numpart.strip()) if p.strip()!='']
    try: parts=[float(p.replace(' ','')) for p in parts]
    except: return np.nan
    if len(parts)==4:  
        return (parts[0]*SQFT['ropani']+parts[1]*SQFT['aana']
                +parts[2]*SQFT['paisa']+parts[3]*SQFT['daam'])
    if len(parts)==1: return parts[0]*SQFT[base]
    return np.nan

# quick check
for ex in ['1-0-0-0 Aana','0-21-0-0 Aana','2500 Sq. Feet','4 Ropani']:
    print(f'{ex:18s} -> {parse_area(ex):,.0f} sqft')

In [ ]:
df = raw.copy()
df['Area_sqft'] = df['Area'].apply(parse_area)

def parse_views(v):
    v=str(v).strip().upper()
    try:
        if v.endswith('K'): return float(v[:-1])*1e3
        if v.endswith('M'): return float(v[:-1])*1e6
        return float(v)
    except: return np.nan
df['Views_n'] = df['Views'].apply(parse_views)

def road_ft(s):
    m=re.search(r'([\d.]+)',str(s))
    if not m: return np.nan
    v=float(m.group(1))
    return v*3.28084 if 'meter' in str(s).lower() else v
df['RoadWidth_ft'] = df['Road Width'].apply(road_ft)

def amen_count(s):
    try: return len(eval(s)) if isinstance(s,str) and s.strip().startswith('[') else 0
    except: return 0
df['Amenities_n'] = df['Amenities'].apply(amen_count)
df['RoadType'] = df['Road Type'].fillna('Unknown')

In [ ]:
valley=['Kathmandu','Lalitpur','Bhaktapur']
steps=[('Raw listings',len(df))]
d=df[df['City'].isin(valley)].copy();                       steps.append(('Valley only',len(d)))
d=d[d['Bedroom']>=1];                                        steps.append(('Houses (Bed>=1)',len(d)))
d=d[d['Area_sqft'].notna()];                                 steps.append(('Parseable area',len(d)))
d=d[(d['Price']>=1_000_000)&(d['Price']<=300_000_000)];      steps.append(('Price 1M-300M',len(d)))
d=d[(d['Area_sqft']>=100)&(d['Area_sqft']<=20000)];          steps.append(('Plausible area',len(d)))
for n,k in steps: print(f'{n:18s}: {k}')

d['Floors']=d['Floors'].fillna(d['Floors'].median())
d['RoadWidth_ft']=d['RoadWidth_ft'].fillna(d['RoadWidth_ft'].median())
d['logPrice']=np.log10(d['Price']); d['logArea']=np.log10(d['Area_sqft'])
print('\nMedian price: NPR {:,.0f}  ({:.2f} crore)'.format(d['Price'].median(), d['Price'].median()/1e7))

## Exploratory Data Analysis

In [ ]:
fig,ax=plt.subplots(1,2,figsize=(11,4))
ax[0].hist(df[(df['Price']>1e5)&(df['Price']<3e8)]['Price']/1e7,bins=60,color=CORAL,alpha=.85)
ax[0].set(title='Raw price (crore NPR) - right skew',xlabel='Price (crore NPR)',ylabel='Listings')
ax[1].hist(d['logPrice'],bins=40,color=TEAL,alpha=.85)
ax[1].set(title='log10(Price) - approx normal',xlabel='log10(Price)')
plt.tight_layout(); plt.show()

In [ ]:
fig,ax=plt.subplots(1,2,figsize=(12,4))
vc=raw['City'].value_counts().head(8)
ax[0].bar(vc.index,vc.values,color=NAVY); ax[0].set_title('Listings by city'); ax[0].tick_params(axis='x',rotation=30)
cp=d.groupby('City')['Price'].median()/1e7
ax[1].bar(cp.index,cp.values,color=[TEAL,CORAL,GOLD]); ax[1].set(title='Median price by district',ylabel='crore NPR')
plt.tight_layout(); plt.show()

In [ ]:
# Price vs area (log-log) and correlation matrix
fig,ax=plt.subplots(1,2,figsize=(13,5))
ax[0].scatter(d['logArea'],d['logPrice'],s=10,alpha=.3,color=TEAL)
m,b=np.polyfit(d['logArea'],d['logPrice'],1); xs=np.linspace(d['logArea'].min(),d['logArea'].max(),50)
ax[0].plot(xs,m*xs+b,color=CORAL,lw=2)
ax[0].set(title=f'Price vs area (r={np.corrcoef(d.logArea,d.logPrice)[0,1]:.2f})',xlabel='log10 area',ylabel='log10 price')
nums=['logPrice','logArea','Bedroom','Bathroom','Floors','Parking','RoadWidth_ft','Amenities_n']
cm=d[nums].corr(); im=ax[1].imshow(cm,cmap='RdYlGn',vmin=-1,vmax=1)
ax[1].set_xticks(range(len(nums))); ax[1].set_xticklabels(nums,rotation=45,ha='right')
ax[1].set_yticks(range(len(nums))); ax[1].set_yticklabels(nums)
for i in range(len(nums)):
    for j in range(len(nums)): ax[1].text(j,i,f'{cm.iloc[i,j]:.2f}',ha='center',va='center',fontsize=7)
ax[1].set_title('Correlation matrix'); plt.colorbar(im,ax=ax[1],fraction=.046)
plt.tight_layout(); plt.show()

## Modelling

In [ ]:
num=['logArea','Bedroom','Bathroom','Floors','Parking','RoadWidth_ft','Amenities_n']
cat=['City','Face','RoadType']
X=d[num+cat]; y=d['logPrice']
Xtr,Xte,ytr,yte=train_test_split(X,y,test_size=0.2,random_state=42)
pre=ColumnTransformer([('num',StandardScaler(),num),
                       ('cat',OneHotEncoder(handle_unknown='ignore'),cat)])

def evaluate(name,model):
    pipe=Pipeline([('pre',pre),('m',model)]).fit(Xtr,ytr)
    p=pipe.predict(Xte)
    medape=np.median(np.abs(10**p-10**yte)/10**yte)*100
    print(f'{name:18s} R2={r2_score(yte,p):.3f}  MedAPE={medape:5.1f}%  '
          f'MAE=NPR {mean_absolute_error(10**yte,10**p):,.0f}')
    return pipe,p

print('Baseline (median) R2=', round(r2_score(yte, np.full(len(yte), ytr.median())),3))
lin,_   = evaluate('LinearRegression', LinearRegression())
rf,rfp  = evaluate('RandomForest', RandomForestRegressor(n_estimators=300,max_depth=14,
                                     min_samples_leaf=3,random_state=42,n_jobs=-1))

In [ ]:
# Predicted vs actual + feature importance
fig,ax=plt.subplots(1,2,figsize=(13,5))
ax[0].scatter(yte,rfp,s=14,alpha=.4,color=TEAL)
lim=[yte.min(),yte.max()]; ax[0].plot(lim,lim,'--',color=CORAL,lw=2)
ax[0].set(title=f'RF predicted vs actual (R2={r2_score(yte,rfp):.2f})',xlabel='actual',ylabel='predicted')
ohe=rf.named_steps['pre'].named_transformers_['cat']
names=num+list(ohe.get_feature_names_out(cat))
imp=pd.Series(rf.named_steps['m'].feature_importances_,index=names).sort_values().tail(12)
ax[1].barh(imp.index,imp.values,color=NAVY); ax[1].set_title('RF feature importance')
plt.tight_layout(); plt.show()